# Long-Document Dataset Sequence-Length Analysis

This notebook captures the Hugging Face / web dataset scouting from 2026-06-14 and visualizes the first-pass sequence-length samples for candidate long-document datasets.

Goal: identify datasets that can force a sliding-window decoder to use concepts as cross-window memory, rather than relying only on local teacher-forced context.

The notebook is designed to run locally from the MrCogito repo root with the project `uv` environment. It uses saved results when available and includes an embedded fallback snapshot so the charts still render.

In [1]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Resolve repo root from playground/<this notebook> when run in Jupyter.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "playground":
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    # This also works when Jupyter is launched from the repo root.
    REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "pyproject.toml").exists() else NOTEBOOK_DIR.parent

SUMMARY_PATH = REPO_ROOT / "Cache" / "Evaluation_reports" / "seqlen_long_candidates_100" / "seqlen_dist_summary.json"
CANDIDATES_PATH = REPO_ROOT / "analysis" / "long_dataset_candidates.json"
SCRIPT_PATH = REPO_ROOT / "analysis" / "dataset_seqlen_distribution.py"

plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print(f"Repo root: {REPO_ROOT}")
print(f"Summary path exists: {SUMMARY_PATH.exists()}")
print(f"Candidate file exists: {CANDIDATES_PATH.exists()}")

Matplotlib is building the font cache; this may take a moment.


Repo root: /Users/ksirg/devel/MrCogito
Summary path exists: True
Candidate file exists: True


In [ ]:
EMBEDDED_RESULTS = [
    {
        "name": "finepdfs_100BT_pdf_pretrain",
        "dataset": "HuggingFaceFW/finepdfs_100BT",
        "subset": "default",
        "split": "train",
        "total_docs": 100,
        "length_source": "token_count column",
        "stats": {"min": 65, "mean": 3768.9, "p50": 1517, "p75": 3146, "p90": 10008, "p95": 15485, "p99": 50569, "max": 50569},
        "bins_pct": {"<=256": 9.0, "<=512": 12.0, "<=1024": 15.0, "<=2048": 28.0, "<=4096": 15.0, "<=8192": 10.0, "<=16384": 7.0, "<=32768": 2.0, ">32768": 2.0},
        "longer_than_pct": {"512": 79.0, "1024": 64.0, "2048": 36.0, "4096": 21.0, "8192": 11.0, "16384": 4.0, "32768": 2.0},
    },
    {
        "name": "longblocks_doc_qa_reasoning",
        "dataset": "utter-project/LongBlocks",
        "subset": "default",
        "split": "train",
        "total_docs": 100,
        "length_source": "tokenizer:HuggingFaceTB/SmolLM2-135M",
        "stats": {"min": 595, "mean": 29492.4, "p50": 4816, "p75": 47542, "p90": 85317, "p95": 90721, "p99": 291622, "max": 291622},
        "bins_pct": {"<=1024": 3.0, "<=2048": 13.0, "<=4096": 29.0, "<=8192": 12.0, "<=32768": 10.0, ">32768": 33.0},
        "longer_than_pct": {"512": 100.0, "1024": 97.0, "2048": 84.0, "4096": 55.0, "8192": 43.0, "16384": 43.0, "32768": 33.0},
    },
    {
        "name": "openmathreasoning_cot",
        "dataset": "nvidia/OpenMathReasoning",
        "subset": "default",
        "split": "cot",
        "total_docs": 100,
        "length_source": "tokenizer:HuggingFaceTB/SmolLM2-135M",
        "stats": {"min": 904, "mean": 7590.1, "p50": 6835, "p75": 12216, "p90": 15473, "p95": 16627, "p99": 18858, "max": 18858},
        "bins_pct": {"<=1024": 3.0, "<=2048": 8.0, "<=4096": 18.0, "<=8192": 32.0, "<=16384": 34.0, "<=32768": 5.0},
        "longer_than_pct": {"512": 100.0, "1024": 97.0, "2048": 89.0, "4096": 71.0, "8192": 39.0, "16384": 5.0, "32768": 0.0},
    },
    {
        "name": "big_reasoning_traces",
        "dataset": "allenai/big-reasoning-traces",
        "subset": "DeepSeek",
        "split": "train",
        "total_docs": 100,
        "length_source": "num_tokens column",
        "stats": {"min": 60, "mean": 3506.7, "p50": 1417, "p75": 3770, "p90": 8673, "p95": 17444, "p99": 23197, "max": 23197},
        "bins_pct": {"<=256": 2.0, "<=512": 2.0, "<=1024": 22.0, "<=2048": 41.0, "<=4096": 11.0, "<=8192": 12.0, "<=16384": 4.0, "<=32768": 6.0},
        "longer_than_pct": {"512": 96.0, "1024": 74.0, "2048": 33.0, "4096": 22.0, "8192": 10.0, "16384": 6.0, "32768": 0.0},
    },
    {
        "name": "openthoughts3_math_code_science",
        "dataset": "open-thoughts/OpenThoughts3-1.2M",
        "subset": "default",
        "split": "train",
        "total_docs": 100,
        "length_source": "tokenizer:HuggingFaceTB/SmolLM2-135M",
        "stats": {"min": 3175, "mean": 18125.8, "p50": 19619, "p75": 20142, "p90": 20804, "p95": 21579, "p99": 23544, "max": 23544},
        "bins_pct": {"<=4096": 1.0, "<=8192": 2.0, "<=16384": 17.0, "<=32768": 80.0},
        "longer_than_pct": {"512": 100.0, "1024": 100.0, "2048": 100.0, "4096": 99.0, "8192": 97.0, "16384": 80.0, "32768": 0.0},
    },
]

CANDIDATE_NOTES = [
    {
        "dataset": "HuggingFaceFW/finepdfs / finepdfs_100BT",
        "scale": "~3T-token parent; 100B-token sample",
        "type": "PDF-derived pretraining documents",
        "why": "Large, varied, longer tail than FineWeb-Edu; good first large-scale long-doc candidate.",
    },
    {
        "dataset": "utter-project/LongBlocks",
        "scale": "57.6k visible HF rows; synthetic long-context SFT",
        "type": "Books, arXiv, Wikipedia, Stack-Edu, StackExchange, FineWeb2-HQ + QA/reasoning responses",
        "why": "Best varied long-context SFT candidate found; many examples are far beyond 32k tokens.",
    },
    {
        "dataset": "nvidia/OpenMathReasoning",
        "scale": "3.2M CoT + 1.7M TIR solutions in full dataset family",
        "type": "Math reasoning traces",
        "why": "Strong structured long reasoning traces, often 4k-16k tokens, but not broad-world knowledge.",
    },
    {
        "dataset": "allenai/big-reasoning-traces",
        "scale": "~2.5B OLMo2 tokens",
        "type": "Compiled reasoning traces",
        "why": "Good mid-training / reasoning data; fewer ultra-long examples than OpenThoughts or LongBlocks.",
    },
    {
        "dataset": "open-thoughts/OpenThoughts3-1.2M",
        "scale": "1.2M rows across math, code, science",
        "type": "Reasoning SFT traces",
        "why": "Very consistently long examples around 16k-24k tokens in the sample.",
    },
    {
        "dataset": "jhu-clsp/ettin-pretraining-data",
        "scale": "1.7T-token varied mixture",
        "type": "DCLM, CC, StarCoder/code, Reddit, peS2o, arXiv, StackExchange, Tulu/FLAN, OpenWebMath, Wikipedia",
        "why": "Very attractive blend, but MDS format; not included in the quick HF streaming sample.",
    },
    {
        "dataset": "bigcode/starcoderdata / the-stack-v2",
        "scale": "~250B StarCoderData; ~900B Stack v2 training tokens",
        "type": "Code files, issues, commits, notebooks, repo-grouped code context",
        "why": "Best large code direction; access/loading is more complex and should be sampled separately.",
    },
]

if SUMMARY_PATH.exists():
    results = json.loads(SUMMARY_PATH.read_text())
    source = f"Loaded from {SUMMARY_PATH.relative_to(REPO_ROOT)}"
else:
    results = EMBEDDED_RESULTS
    source = "Using embedded 2026-06-14 fallback sample"

print(source)
print(f"Loaded {len(results)} sampled candidates")

In [ ]:
def markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(col, "")).replace("\n", "<br>") for col in columns) + " |")
    return "\n".join([header, sep, *body])

# Show scouting notes beyond just the five measured datasets.
display(Markdown("## Candidate Scout Notes\n" + markdown_table(CANDIDATE_NOTES, ["dataset", "scale", "type", "why"])))

In [ ]:
def short_name(name: str) -> str:
    return {
        "finepdfs_100BT_pdf_pretrain": "FinePDFs-100BT",
        "longblocks_doc_qa_reasoning": "LongBlocks",
        "openmathreasoning_cot": "OpenMathReasoning",
        "big_reasoning_traces": "BigReasoningTraces",
        "openthoughts3_math_code_science": "OpenThoughts3",
    }.get(name, name)

summary_rows = []
for result in results:
    stats = result["stats"]
    summary_rows.append({
        "name": short_name(result["name"]),
        "dataset": result["dataset"],
        "n": result["total_docs"],
        "length_source": result["length_source"],
        "p50": stats["p50"],
        "p90": stats["p90"],
        "p95": stats["p95"],
        "max": stats["max"],
        ">4k%": result["longer_than_pct"].get("4096", 0),
        ">8k%": result["longer_than_pct"].get("8192", 0),
        ">16k%": result["longer_than_pct"].get("16384", 0),
        ">32k%": result["longer_than_pct"].get("32768", 0),
    })

display(Markdown("## 100-Row Sample Summary\n" + markdown_table(summary_rows, ["name", "dataset", "n", "length_source", "p50", "p90", "p95", "max", ">4k%", ">8k%", ">16k%", ">32k%"])) )

In [ ]:
names = [short_name(r["name"]) for r in results]
percentiles = ["p50", "p90", "p95"]
width = 0.24
x = list(range(len(names)))

fig, ax = plt.subplots(figsize=(12, 6))
for i, pct in enumerate(percentiles):
    values = [r["stats"][pct] for r in results]
    offsets = [v + (i - 1) * width for v in x]
    ax.bar(offsets, values, width=width, label=pct)

ax.axhline(4096, color="tab:gray", linestyle="--", linewidth=1, label="4k")
ax.axhline(8192, color="tab:orange", linestyle="--", linewidth=1, label="8k")
ax.axhline(32768, color="tab:red", linestyle="--", linewidth=1, label="32k")
ax.set_yscale("log")
ax.set_ylabel("Sequence length (tokens, log scale)")
ax.set_title("Candidate Dataset Length Percentiles")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right")
ax.legend(ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
thresholds = [512, 1024, 2048, 4096, 8192, 16384, 32768]

fig, ax = plt.subplots(figsize=(12, 6))
for result in results:
    y = [result["longer_than_pct"].get(str(t), 0.0) for t in thresholds]
    ax.plot(thresholds, y, marker="o", label=short_name(result["name"]))

ax.set_xscale("log", base=2)
ax.set_xticks(thresholds)
ax.set_xticklabels([f">{t//1024}k" if t >= 1024 else f">{t}" for t in thresholds])
ax.set_ylim(0, 105)
ax.set_ylabel("Examples longer than threshold (%)")
ax.set_xlabel("Context threshold")
ax.set_title("Long-Range Forcing Potential")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
BIN_ORDER = ["<=256", "<=512", "<=1024", "<=2048", "<=4096", "<=8192", "<=16384", "<=32768", ">32768"]

fig, ax = plt.subplots(figsize=(13, 7))
y_positions = list(range(len(results)))
left = [0.0] * len(results)

for bin_label in BIN_ORDER:
    values = [r.get("bins_pct", {}).get(bin_label, 0.0) for r in results]
    if not any(values):
        continue
    ax.barh(y_positions, values, left=left, label=bin_label)
    left = [l + v for l, v in zip(left, values)]

ax.set_yticks(y_positions)
ax.set_yticklabels([short_name(r["name"]) for r in results])
ax.set_xlabel("Share of sampled examples (%)")
ax.set_title("Length-Bin Distribution")
ax.legend(title="Token bin", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Optional: rerun the sampler locally.
# This uses the reusable script and candidate file created during the analysis.
# It is CPU-only. Larger max_docs values improve percentile stability but download/stream more data.

RUN_FRESH_SAMPLE = False
MAX_DOCS = 500  # try 100 first; 1_000-10_000 is better for stable tails
OUT_DIR = REPO_ROOT / "Cache" / "Evaluation_reports" / f"seqlen_long_candidates_{MAX_DOCS}"
TOKENIZER = "HuggingFaceTB/SmolLM2-135M"

cmd = [
    "uv", "run", "python", str(SCRIPT_PATH),
    "--candidates", str(CANDIDATES_PATH),
    "--tokenizer", TOKENIZER,
    "--max_docs", str(MAX_DOCS),
    "--out_dir", str(OUT_DIR),
]

print(" ".join(cmd))

if RUN_FRESH_SAMPLE:
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
    fresh_summary = OUT_DIR / "seqlen_dist_summary.json"
    print(f"Fresh summary written to: {fresh_summary}")
else:
    print("Set RUN_FRESH_SAMPLE = True to execute this command.")

## Interpretation

For a sliding-window decoder with last-K local context, useful training data needs coherent examples that extend well beyond K. In this first-pass sample:

- **OpenThoughts3** is consistently long: almost every sampled example is above 8k tokens, but it is SFT/reasoning-trace data rather than a 100B-token pretraining corpus.
- **LongBlocks** is the best varied long-context SFT candidate: it mixes books, arXiv, Wikipedia, Stack-Edu, StackExchange, FineWeb2-HQ, and synthetic QA/reasoning responses. The tail is very long, with many examples beyond 32k tokens.
- **OpenMathReasoning** has strong 4k-16k structured reasoning examples, useful for reasoning pressure but domain-narrow.
- **FinePDFs-100BT** is the strongest large-scale pretraining candidate in this measured set: it is varied and has a real long tail, though most examples are still below 8k.
- **Big reasoning traces** are useful but have a weaker ultra-long tail in this sample.

Caveats:

- The sample size here is only 100 rows per candidate; rerun with 1k-10k rows before making a training-data decision.
- FinePDFs uses its provided `token_count`; other tokenized measurements use `HuggingFaceTB/SmolLM2-135M`.
- `peS2o` full papers remain attractive, but the current `datasets` package rejects its old dataset script. It needs separate conversion/download handling.
- `jhu-clsp/ettin-pretraining-data`, `bigcode/the-stack-v2`, and long-context packed corpora are important follow-ups, but they need custom loaders or gated/data-format-specific access.